# Data Analysis and Visualisation

## Lab 8: Check the claims

### Preliminaries

In [ ]:
import pandas as pd

# Load the table, parsing the dates
date_columns = ["first_review", "last_review"]
listings = pd.read_csv("data/listings_clean.csv", parse_dates=date_columns)

### Claim 1: "A night in Stockholm costs 1,791."

In [ ]:
# The price is in the column "price". How many listings have a value in it?
price = listings["price"]
print("Listings with price:", price.notna().sum(), round(price.notna().mean() * 100, 1))

# The number in the claim: the median of those listings.
print("Median price for listings with price:", price.median())

Listings with price: 2957 64.6
Median price for listings with price: 1791.0


In [ ]:
# How many listings have no price, and what share of all listings are they?
print("Listings without price and their share:", price.isna().sum(), round(price.isna().mean() * 100, 1))

# What do the listings without a price have in common? Compare to the days they are open.
no_price = listings[price.isna()]
print("Listings with no availability:", (no_price["availability_365"] == 0).sum())

Listings without price and their share: 1617 35.4
Listings with no availability: 1490


In [ ]:
# Compare long minimum stays with the others, among the listings that have a price.
priced = listings[price.notna()]
long_stay = priced["minimum_nights"] >= 28
print(long_stay.sum(), priced.loc[long_stay, "price"].median())
print((~long_stay).sum(), priced.loc[~long_stay, "price"].median())

97 708.89
2860 1839.25


**Verdict:** Holds with reservations. The number is the median of the 2957 listings that have a price. The other 1617 (35.4 %) have none, and 1490 of them have no open days (availability 0), so they are probably not for rent right now. The 97 listings that require 28 nights or more have a median of 708.9, against 1839.2 for the other 2860, so "a night" depends a lot on the minimum stay.

**What the sentence doesn't say:** That a third of the listings are missing, that it is a median and not a price anyone is guaranteed to pay, and it doesn't mention the currency.

**New sentence:** "At the time data was collected the median price per night was 1,791 SEK for the 2,957 Airbnb listings in Stockholm that had a price, which is 65 percent of all listings."

### Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."

In [ ]:
# Only listings with a rating can have a perfect one
rated = listings[listings["has_reviews"]]
print("Lisings with reviews:", len(rated))

Lisings with reviews: 3930


In [ ]:
# How many of the rated listings have a perfect 5.00, and what share is that?
perfect = rated["review_scores_rating"] == 5
print("Listings with perfect reviews:", perfect.sum(), round(perfect.mean() * 100, 1))

Listings with perfect reviews: 1484 37.8


In [ ]:
# How many of the perfect ones have at least 20 reviews?
many = rated["number_of_reviews"] >= 20
print((perfect & many).sum(), round((perfect & many).sum() / perfect.sum() * 100, 1))

130 8.8


In [ ]:
# Median number of reviews: perfect listings against the other rated listings.
print("Median # of reviews for perfect listings:", rated.loc[perfect, "number_of_reviews"].median())
print("Median # of reviews for non-perfect listings:", rated.loc[~perfect, "number_of_reviews"].median())

Median # of reviews for perfect listings: 4.0
Median # of reviews for non-perfect listings: 25.0


**Verdict:** Does not hold. 1484 of the 3930 rated listings (37.8 %) have a perfect 5.00, but only 130 of them (8.8 %) have at least 20 reviews. The median is 4 reviews for the perfect listings and 25 for the others. A perfect rating mostly means that a few guests were satisfied.

**What the sentence doesn't say:** "Best" isn't defined at all, and the listings without a rating are left out.

**New sentence:** "38 percent of the rated listings have a perfect 5.00, but 91 percent of them have fewer than 20 reviews, so the rating says little about which are best."

### Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in 2024."

In [ ]:
# The year of the first review, counted per year.
year = listings["first_review"].dt.year
display(year.value_counts().sort_index())
print("Listings without reviews:", year.isna().sum())

first_review
2011.0      2
2012.0     12
2013.0     26
2014.0     51
2015.0    116
2016.0    131
2017.0    148
2018.0    191
2019.0    234
2020.0     96
2021.0    159
2022.0    332
2023.0    565
2024.0    610
2025.0    542
2026.0    715
Name: count, dtype: int64

Listings without reviews: 644


In [ ]:
# The share of listings from 2016 and 2024 that were reviewed again in 2026.
for yr in [2016, 2024]:
    group = listings[year == yr]
    print(f"Listings from year {yr} reviewed again in 2026:", len(group), round((group["last_review"].dt.year == 2026).mean(), 2))

Listings from year 2016 reviewed again in 2026: 131 0.53
Listings from year 2024 reviewed again in 2026: 610 0.67


**Verdict:** Cannot be concluded from this table. The counts are right (131 and 610), but the table only holds listings that exist today, so listings that started in 2016 and have since closed are missing. Of the 2016 listings, 53% were reviewed in 2026, against 67% of the 2024 listings, so the older group has lost more. This is survivorship bias: the table shows who is left, not who was there. The counts also come from the first review, which is later than the day the listing started, and the 644 listings without any review have no year at all.

**New sentence:** "Of the listings that still exist, 131 got their first review in 2016 and 610 in 2024. We cannot say how many were started in those years."

### Claim 4:  "Most hosts in Stockholm are professional companies with many listings."

In [ ]:
# Count how many different hosts are there, compared with the number of listings?
print("Total number of listings:", len(listings))
print("Number of hosts:", listings["host_id"].nunique())

Total number of listings: 4574
Number of hosts: 3591


In [ ]:
# Count how many listings does each host have?
per_host = listings["host_id"].value_counts()

# Hosts with exactly one listing.
one = per_host == 1
print("Hosts with one listing:", one.sum(), "=", round(one.mean() * 100, 1), "%")

# Hosts with two or more: how many, and how many listings they own together.
several = per_host[per_host >= 2]
print("Hosts with several listings:", len(several), "=", round(len(several) / per_host.size * 100, 1), "%")
print("Listings these hosts own together:", several.sum(), "=", round(several.sum() / len(listings) * 100, 1), "%")

Hosts with one listing: 3237 = 90.1 %
Hosts with several listings: 354 = 9.9 %
Listings these hosts own together: 1337 = 29.2 %


In [ ]:
# The largest host.
print(per_host.max())

49


**Verdict:** Does not hold (and "company" cannot be concluded). The 4574 listings belong to 3591 hosts. 3237 of them (90.1 %) have one listing, and the 354 hosts with two or more (9.9 %) own 1337 listings, 29.2 % of the total. The largest host has 49. There is no column that says whether a host is a company.

**What the sentence doesn't say:** "most hosts" is false, but "most listings" would be false too (29 %). A host with many listings could also be a private person.

**New sentence:** "90 percent of the hosts have only one listing, but the 10 percent with two or more own 29 percent of all listings."